# 09 - Machine Learning: weekly production forecast

**Question this model answers:** how many units will the plant actually
produce next week, by process? This helps plan raw-material purchases
and shipments against a realistic expectation, not just the plan.

I use a **Random Forest** (many decision trees combined), because it
handles non-linear patterns well without needing the data to be scaled first.


In [ ]:
import sys
sys.path.insert(0, "../machine_learning")

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
import ml_lib as ml

PROCESSED = "../../datasets/processed"
MODELS = "../../models"
REPORTS = "../../reports"
os.makedirs(MODELS, exist_ok=True)

production = pd.read_csv(f"{PROCESSED}/fact_production_processed.csv", encoding="utf-8-sig", parse_dates=["Date"])
plan = pd.read_csv(f"{PROCESSED}/fact_production_plan_processed.csv", encoding="utf-8-sig", parse_dates=["Date"])


## Step 1: build the weekly table

One row per (Process, week). Combining all four processes into a
single table, with `Process` as a categorical variable, gives far more
training rows than training a separate model per process.


In [ ]:
def week_start(date_column):
    iso_weekday = date_column.dt.isocalendar().day
    return date_column - pd.to_timedelta(iso_weekday - 1, unit="D")


production["WeekStart"] = week_start(production["Date"])
plan["WeekStart"] = week_start(plan["Date"])

weekly_production = production.groupby(["Process", "WeekStart"]).agg(
    ProducedQty=("ProducedQty", "sum"),
    Availability=("Availability", "mean"),
    Performance=("Performance", "mean"),
    Quality=("Quality", "mean"),
    OrderCount=("WorkOrder", "count"),
).reset_index()

weekly_plan = plan.groupby(["Process", "WeekStart"])["PlannedQty"].sum().reset_index()

table = weekly_production.merge(weekly_plan, on=["Process", "WeekStart"], how="left")
table = table.sort_values(["Process", "WeekStart"]).reset_index(drop=True)
table["ISOWeek"] = table["WeekStart"].dt.isocalendar().week.astype(int)

print(f"Weekly table: {len(table)} rows ({table['Process'].nunique()} processes)")
table.head()


## Step 2: lag features

A forecast for week *t* can only use information that existed **before**
week *t* -- so I use `shift(1)`/`shift(2)` (production from 1 and 2
weeks ago), computed separately per process so one process's history
doesn't leak into another.


In [ ]:
table = table.sort_values(["Process", "WeekStart"])
production_by_process = table.groupby("Process")["ProducedQty"]

table["Lag1_ProducedQty"] = production_by_process.shift(1)
table["Lag2_ProducedQty"] = production_by_process.shift(2)
table["RollingMean4_ProducedQty"] = (
    production_by_process.shift(1).rolling(4, min_periods=2).mean().reset_index(level=0, drop=True)
)
table["Lag1_Availability"] = table.groupby("Process")["Availability"].shift(1)

# the first few weeks of each process don't have enough history yet -- drop them
model_data = table.dropna(subset=["Lag1_ProducedQty", "Lag2_ProducedQty", "RollingMean4_ProducedQty"]).copy()

# turns the text column "Process" into 0/1 columns, one per process
model_data = pd.get_dummies(model_data, columns=["Process"], prefix="Process")

feature_columns = [c for c in model_data.columns if c.startswith("Process_")] + [
    "PlannedQty", "ISOWeek", "Lag1_ProducedQty", "Lag2_ProducedQty", "RollingMean4_ProducedQty", "Lag1_Availability",
]
target_column = "ProducedQty"

print(f"Model data: {len(model_data)} rows, {len(feature_columns)} features")


## Step 3: train/test split and a simple baseline

The simple baseline ("next week will match the 4-week rolling average")
is the bar the model needs to clear to be worth using.


In [ ]:
train, test = ml.split_by_date(model_data, "WeekStart", test_fraction=0.2)
X_train, y_train = train[feature_columns], train[target_column]
X_test, y_test = test[feature_columns], test[target_column]

print(f"Train: {len(train)} rows ({train['WeekStart'].min().date()} to {train['WeekStart'].max().date()})")
print(f"Test:  {len(test)} rows ({test['WeekStart'].min().date()} to {test['WeekStart'].max().date()})")

baseline_prediction = test["RollingMean4_ProducedQty"]
print("\nSimple baseline (4-week rolling average):")
print(ml.regression_metrics(y_test, baseline_prediction))


## Step 4: train the model

In [ ]:
model = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
model.fit(X_train, y_train)

prediction = model.predict(X_test)

print("Random Forest:")
print(ml.regression_metrics(y_test, prediction))


## Step 5: actual vs. predicted chart, and feature importance

In [ ]:
test = test.copy()
test["Predicted"] = prediction

fig, ax = plt.subplots(figsize=(11, 5))
for process_column in [c for c in model_data.columns if c.startswith("Process_")]:
    process_name = process_column.replace("Process_", "")
    rows = test[test[process_column]].sort_values("WeekStart")
    ax.plot(rows["WeekStart"], rows["ProducedQty"], marker="o", label=f"{process_name} (actual)", linewidth=1)
    ax.plot(rows["WeekStart"], rows["Predicted"], marker="x", linestyle="--", label=f"{process_name} (predicted)", linewidth=1)
ax.set_ylabel("Units produced (weekly)")
ax.set_title("Weekly production forecast: actual vs. predicted (test period)")
ax.legend(fontsize=7, ncol=2)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(f"{REPORTS}/35_ml_production_forecast_actual_vs_predicted.png", dpi=150)
plt.show()


In [ ]:
importance = pd.Series(model.feature_importances_, index=feature_columns).sort_values()

fig, ax = plt.subplots(figsize=(7, 5))
importance.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Feature importance -- production forecast model")
plt.tight_layout()
plt.savefig(f"{REPORTS}/36_ml_production_forecast_feature_importance.png", dpi=150)
plt.show()

ml.save_model(
    model, f"{MODELS}/production_forecast_model.pkl",
    feature_cols=feature_columns, target_col=target_column, model_name="Random Forest",
    trained_on=f"{train['WeekStart'].min().date()} to {train['WeekStart'].max().date()}",
    test_metrics=ml.regression_metrics(y_test, test["Predicted"]),
)
print(f"Model saved to {MODELS}/production_forecast_model.pkl")


## Step 6: the actual forecast for next week

In [ ]:
last_week = table.sort_values("WeekStart").groupby("Process").tail(1).copy()
last_week_features = pd.get_dummies(last_week, columns=["Process"], prefix="Process")

for column in feature_columns:
    if column not in last_week_features.columns:
        last_week_features[column] = 0

# advances the lags by one week, using what we already know
last_week_features["Lag2_ProducedQty"] = last_week_features["Lag1_ProducedQty"]
last_week_features["Lag1_ProducedQty"] = last_week["ProducedQty"].values
last_week_features["RollingMean4_ProducedQty"] = last_week["RollingMean4_ProducedQty"].values
last_week_features["PlannedQty"] = last_week["PlannedQty"].values  # assumes next week's plan follows the current pace
last_week_features["ISOWeek"] = (last_week["ISOWeek"] + 1) % 53

next_week_prediction = model.predict(last_week_features[feature_columns])

forecast_table = pd.DataFrame({
    "Process": last_week["Process"].values,
    "ForecastedProducedQty": next_week_prediction.round(0).astype(int),
})
print(forecast_table)
print(f"\nTotal plant-wide forecast for next week: {forecast_table['ForecastedProducedQty'].sum():,} units")


## Step 7: export the results

In [ ]:
forecast_table["GeneratedFrom"] = last_week["WeekStart"].max().date().isoformat()
forecast_table.to_csv(f"{PROCESSED}/ml_predictions_production_forecast.csv", index=False, encoding="utf-8-sig")

# also export the test-period history (actual vs. predicted), so a
# dashboard can show how well the model tracked over time
process_columns = [c for c in test.columns if c.startswith("Process_")]
test["Process"] = test[process_columns].idxmax(axis=1).str.replace("Process_", "")

history = test[["WeekStart", "Process", "ProducedQty", "Predicted"]].rename(columns={
    "ProducedQty": "ActualProducedQty", "Predicted": "PredictedProducedQty",
})
history.to_csv(f"{PROCESSED}/ml_predictions_production_forecast_history.csv", index=False, encoding="utf-8-sig")

print("Exported: ml_predictions_production_forecast.csv and ..._history.csv")


## Part 2: forecasting weekly downtime hours by process

**Question this second model answers:** how many hours of unplanned
downtime should each process expect next week? Paired with the
production forecast above, this gives a "production hours vs. stopped
hours" view for the coming week -- useful for capacity planning and for
deciding where a maintenance team should focus preventive work.

Same approach as before: weekly grain, one row per (process, week),
lag features only (never looking at the future), Random Forest,
date-based train/test split.


In [ ]:
downtime = pd.read_csv(f"{PROCESSED}/fact_downtime_processed.csv", encoding="utf-8-sig", parse_dates=["Date"])
downtime["WeekStart"] = week_start(downtime["Date"])

unplanned = downtime[downtime["PlannedStoppage"] == "No"]
weekly_downtime = (
    unplanned.groupby(["Process", "WeekStart"])["DowntimeDurationMin"].sum() / 60
).reset_index().rename(columns={"DowntimeDurationMin": "DowntimeHours"})

downtime_table = table[["Process", "WeekStart", "ISOWeek"]].merge(
    weekly_downtime, on=["Process", "WeekStart"], how="left"
).fillna({"DowntimeHours": 0})
downtime_table = downtime_table.sort_values(["Process", "WeekStart"])

print(f"Weekly downtime table: {len(downtime_table)} rows")
downtime_table.head()


In [ ]:
downtime_by_process = downtime_table.groupby("Process")["DowntimeHours"]

downtime_table["Lag1_DowntimeHours"] = downtime_by_process.shift(1)
downtime_table["Lag2_DowntimeHours"] = downtime_by_process.shift(2)
downtime_table["RollingMean4_DowntimeHours"] = (
    downtime_by_process.shift(1).rolling(4, min_periods=2).mean().reset_index(level=0, drop=True)
)

downtime_model_data = downtime_table.dropna(
    subset=["Lag1_DowntimeHours", "Lag2_DowntimeHours", "RollingMean4_DowntimeHours"]
).copy()
downtime_model_data = pd.get_dummies(downtime_model_data, columns=["Process"], prefix="Process")

downtime_feature_columns = [c for c in downtime_model_data.columns if c.startswith("Process_")] + [
    "ISOWeek", "Lag1_DowntimeHours", "Lag2_DowntimeHours", "RollingMean4_DowntimeHours",
]
downtime_target_column = "DowntimeHours"

print(f"Downtime model data: {len(downtime_model_data)} rows, {len(downtime_feature_columns)} features")


In [ ]:
downtime_train, downtime_test = ml.split_by_date(downtime_model_data, "WeekStart", test_fraction=0.2)
X_train_dt, y_train_dt = downtime_train[downtime_feature_columns], downtime_train[downtime_target_column]
X_test_dt, y_test_dt = downtime_test[downtime_feature_columns], downtime_test[downtime_target_column]

downtime_baseline = downtime_test["RollingMean4_DowntimeHours"]
print("Simple baseline (4-week rolling average):")
print(ml.regression_metrics(y_test_dt, downtime_baseline))

downtime_model = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
downtime_model.fit(X_train_dt, y_train_dt)
downtime_prediction = downtime_model.predict(X_test_dt)

print("\nRandom Forest:")
print(ml.regression_metrics(y_test_dt, downtime_prediction))


In [ ]:
downtime_test = downtime_test.copy()
downtime_test["Predicted"] = downtime_prediction

fig, ax = plt.subplots(figsize=(11, 5))
for process_column in [c for c in downtime_model_data.columns if c.startswith("Process_")]:
    process_name = process_column.replace("Process_", "")
    rows = downtime_test[downtime_test[process_column]].sort_values("WeekStart")
    ax.plot(rows["WeekStart"], rows["DowntimeHours"], marker="o", label=f"{process_name} (actual)", linewidth=1)
    ax.plot(rows["WeekStart"], rows["Predicted"], marker="x", linestyle="--", label=f"{process_name} (predicted)", linewidth=1)
ax.set_ylabel("Unplanned downtime hours (weekly)")
ax.set_title("Weekly downtime forecast: actual vs. predicted (test period)")
ax.legend(fontsize=7, ncol=2)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(f"{REPORTS}/45_ml_downtime_forecast_actual_vs_predicted.png", dpi=150)
plt.show()

ml.save_model(
    downtime_model, f"{MODELS}/downtime_forecast_model.pkl",
    feature_cols=downtime_feature_columns, target_col=downtime_target_column, model_name="Random Forest",
    test_metrics=ml.regression_metrics(y_test_dt, downtime_test["Predicted"]),
)
print(f"Model saved to {MODELS}/downtime_forecast_model.pkl")


In [ ]:
last_week_dt = downtime_table.sort_values("WeekStart").groupby("Process").tail(1).copy()
last_week_dt_features = pd.get_dummies(last_week_dt, columns=["Process"], prefix="Process")

for column in downtime_feature_columns:
    if column not in last_week_dt_features.columns:
        last_week_dt_features[column] = 0

last_week_dt_features["Lag2_DowntimeHours"] = last_week_dt_features["Lag1_DowntimeHours"]
last_week_dt_features["Lag1_DowntimeHours"] = last_week_dt["DowntimeHours"].values
last_week_dt_features["RollingMean4_DowntimeHours"] = last_week_dt["RollingMean4_DowntimeHours"].values
last_week_dt_features["ISOWeek"] = (last_week_dt["ISOWeek"] + 1) % 53

next_week_downtime = downtime_model.predict(last_week_dt_features[downtime_feature_columns])

downtime_forecast_table = pd.DataFrame({
    "Process": last_week_dt["Process"].values,
    "ForecastedDowntimeHours": next_week_downtime.round(1),
})

# joins the two forecasts side by side: production hours vs. downtime hours, next week, by process
combined_forecast = forecast_table.merge(downtime_forecast_table, on="Process", how="left")
print(combined_forecast)


### Exporting the combined forecast

In [ ]:
downtime_forecast_table["GeneratedFrom"] = last_week_dt["WeekStart"].max().date().isoformat()
downtime_forecast_table.to_csv(f"{PROCESSED}/ml_predictions_downtime_forecast.csv", index=False, encoding="utf-8-sig")

downtime_process_columns = [c for c in downtime_test.columns if c.startswith("Process_")]
downtime_test["Process"] = downtime_test[downtime_process_columns].idxmax(axis=1).str.replace("Process_", "")
downtime_history = downtime_test[["WeekStart", "Process", "DowntimeHours", "Predicted"]].rename(columns={
    "DowntimeHours": "ActualDowntimeHours", "Predicted": "PredictedDowntimeHours",
})
downtime_history.to_csv(f"{PROCESSED}/ml_predictions_downtime_forecast_history.csv", index=False, encoding="utf-8-sig")

print("Exported: ml_predictions_downtime_forecast.csv and ..._history.csv")


## Part 3: forecasting weekly rejected units by process

**Question this third model answers:** how many units are likely to be
rejected next week, by process? Paired with the production forecast
(Part 1), this gives a "produced vs. rejected" view for the coming
week -- this is a different question from the scrap **rate** model in
notebook `10` (which predicts the % risk of a single upcoming order),
since this one forecasts the **absolute weekly volume** the plant
should expect to scrap, useful for material and disposal planning.


In [ ]:
weekly_rejected = production.groupby(["Process", "WeekStart"])["RejectedQty"].sum().reset_index()

rejected_table = table[["Process", "WeekStart", "ISOWeek", "ProducedQty"]].merge(
    weekly_rejected, on=["Process", "WeekStart"], how="left"
).fillna({"RejectedQty": 0})
rejected_table = rejected_table.sort_values(["Process", "WeekStart"])

rejected_by_process = rejected_table.groupby("Process")["RejectedQty"]
rejected_table["Lag1_RejectedQty"] = rejected_by_process.shift(1)
rejected_table["Lag2_RejectedQty"] = rejected_by_process.shift(2)
rejected_table["RollingMean4_RejectedQty"] = (
    rejected_by_process.shift(1).rolling(4, min_periods=2).mean().reset_index(level=0, drop=True)
)

rejected_model_data = rejected_table.dropna(
    subset=["Lag1_RejectedQty", "Lag2_RejectedQty", "RollingMean4_RejectedQty"]
).copy()
rejected_model_data = pd.get_dummies(rejected_model_data, columns=["Process"], prefix="Process")

rejected_feature_columns = [c for c in rejected_model_data.columns if c.startswith("Process_")] + [
    "ISOWeek", "ProducedQty", "Lag1_RejectedQty", "Lag2_RejectedQty", "RollingMean4_RejectedQty",
]
rejected_target_column = "RejectedQty"

print(f"Rejected-units model data: {len(rejected_model_data)} rows, {len(rejected_feature_columns)} features")


In [ ]:
rejected_train, rejected_test = ml.split_by_date(rejected_model_data, "WeekStart", test_fraction=0.2)
X_train_rj, y_train_rj = rejected_train[rejected_feature_columns], rejected_train[rejected_target_column]
X_test_rj, y_test_rj = rejected_test[rejected_feature_columns], rejected_test[rejected_target_column]

rejected_baseline = rejected_test["RollingMean4_RejectedQty"]
print("Simple baseline (4-week rolling average):")
print(ml.regression_metrics(y_test_rj, rejected_baseline))

rejected_model = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
rejected_model.fit(X_train_rj, y_train_rj)
rejected_prediction = rejected_model.predict(X_test_rj)

print("\nRandom Forest:")
print(ml.regression_metrics(y_test_rj, rejected_prediction))


In [ ]:
rejected_test = rejected_test.copy()
rejected_test["Predicted"] = rejected_prediction

fig, ax = plt.subplots(figsize=(11, 5))
for process_column in [c for c in rejected_model_data.columns if c.startswith("Process_")]:
    process_name = process_column.replace("Process_", "")
    rows = rejected_test[rejected_test[process_column]].sort_values("WeekStart")
    ax.plot(rows["WeekStart"], rows["RejectedQty"], marker="o", label=f"{process_name} (actual)", linewidth=1)
    ax.plot(rows["WeekStart"], rows["Predicted"], marker="x", linestyle="--", label=f"{process_name} (predicted)", linewidth=1)
ax.set_ylabel("Rejected units (weekly)")
ax.set_title("Weekly rejected-units forecast: actual vs. predicted (test period)")
ax.legend(fontsize=7, ncol=2)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(f"{REPORTS}/46_ml_rejected_forecast_actual_vs_predicted.png", dpi=150)
plt.show()

ml.save_model(
    rejected_model, f"{MODELS}/rejected_forecast_model.pkl",
    feature_cols=rejected_feature_columns, target_col=rejected_target_column, model_name="Random Forest",
    test_metrics=ml.regression_metrics(y_test_rj, rejected_test["Predicted"]),
)
print(f"Model saved to {MODELS}/rejected_forecast_model.pkl")


In [ ]:
last_week_rj = rejected_table.sort_values("WeekStart").groupby("Process").tail(1).copy()
last_week_rj_features = pd.get_dummies(last_week_rj, columns=["Process"], prefix="Process")

for column in rejected_feature_columns:
    if column not in last_week_rj_features.columns:
        last_week_rj_features[column] = 0

last_week_rj_features["Lag2_RejectedQty"] = last_week_rj_features["Lag1_RejectedQty"]
last_week_rj_features["Lag1_RejectedQty"] = last_week_rj["RejectedQty"].values
last_week_rj_features["RollingMean4_RejectedQty"] = last_week_rj["RollingMean4_RejectedQty"].values
last_week_rj_features["ProducedQty"] = forecast_table.set_index("Process").reindex(last_week_rj["Process"])["ForecastedProducedQty"].values
last_week_rj_features["ISOWeek"] = (last_week_rj["ISOWeek"] + 1) % 53

next_week_rejected = rejected_model.predict(last_week_rj_features[rejected_feature_columns])

rejected_forecast_table = pd.DataFrame({
    "Process": last_week_rj["Process"].values,
    "ForecastedRejectedQty": next_week_rejected.round(0).astype(int),
})

# the full three-way combined forecast for next week: produced, downtime hours, rejected
full_combined_forecast = combined_forecast.merge(rejected_forecast_table, on="Process", how="left")
print(full_combined_forecast)


In [ ]:
rejected_forecast_table["GeneratedFrom"] = last_week_rj["WeekStart"].max().date().isoformat()
rejected_forecast_table.to_csv(f"{PROCESSED}/ml_predictions_rejected_forecast.csv", index=False, encoding="utf-8-sig")

rejected_process_columns = [c for c in rejected_test.columns if c.startswith("Process_")]
rejected_test["Process"] = rejected_test[rejected_process_columns].idxmax(axis=1).str.replace("Process_", "")
rejected_history = rejected_test[["WeekStart", "Process", "RejectedQty", "Predicted"]].rename(columns={
    "RejectedQty": "ActualRejectedQty", "Predicted": "PredictedRejectedQty",
})
rejected_history.to_csv(f"{PROCESSED}/ml_predictions_rejected_forecast_history.csv", index=False, encoding="utf-8-sig")

print("Exported: ml_predictions_rejected_forecast.csv and ..._history.csv")
